# Integrated Political Bias Detection Pipeline

This notebook integrates **XLNet** for attribute encoding (Ideology, Topic, Issue) and **CLIP** for news content feature extraction, combined with a **Cross-Attention Feature Fusion** mechanism.

### Overview:
1. **Verbatim Components**: Direct implementations from original research notebooks.
2. **Cross-Attention Fusion**: Headlines act as Query, attributes as Key/Value.
3. **Unified Pipeline**: Training, Validation, and Testing with visualization logic.

In [1]:
import torch
import torch.nn as nn
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from torch.utils.data import Dataset, DataLoader
from transformers import XLNetTokenizer, XLNetModel, CLIPTokenizer, CLIPTextModel
from sklearn.metrics import classification_report, confusion_matrix
import logging
import os

logging.basicConfig(level=logging.INFO, format="%(asctime)s | %(levelname)s | %(message)s", datefmt="%H:%M:%S")
log = logging.getLogger(__name__)

## 1. Verbatim Components from Notebooks

In [2]:
class LinearProjectionBlock(nn.Module):
    def __init__(self, input_dim, hidden_dim, output_dim, dropout_rate=0.5):
        super().__init__()
        layers = []
        in_dim = input_dim
        for h in hidden_dim:
            layers.append(nn.Linear(in_dim,h))
            layers.append(nn.BatchNorm1d(h))
            layers.append(nn.ReLU(inplace = True))
            layers.append(nn.Dropout(p = dropout_rate))
            in_dim = h
        layers.append(nn.Linear(in_dim, output_dim))
        layers.append(nn.BatchNorm1d(output_dim))
        self.projection = nn.Sequential(*layers)
    def forward(self,x):
        return self.projection(x)

class LinearTextProjection(nn.Module):
    def __init__(self, clip_dim=512, proj_dim=256, dropout=0.1):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(clip_dim, proj_dim),
            nn.LayerNorm(proj_dim),
            nn.GELU(),
            nn.Dropout(dropout),
        )
    def forward(self, x):
        return self.net(x)

class NewsFeatureExtractor(nn.Module):
    def __init__(self, clip_model_name="openai/clip-vit-base-patch32", proj_dim=256, dropout=0.1, freeze_clip=True):
        super().__init__()
        self.clip_encoder = CLIPTextModel.from_pretrained(clip_model_name)
        if freeze_clip:
            for p in self.clip_encoder.parameters():
                p.requires_grad_(False)
        clip_dim = self.clip_encoder.config.hidden_size
        self.projection = LinearTextProjection(clip_dim=clip_dim, proj_dim=proj_dim, dropout=dropout)
    def forward(self, input_ids, attention_mask):
        clip_out = self.clip_encoder(input_ids=input_ids, attention_mask=attention_mask)
        pooled   = clip_out.pooler_output
        return self.projection(pooled)

def build_clip_prompt(text: str, max_words: int = 60) -> str:
    words = text.strip().split()[:max_words]
    return f"a photo of v* {' '.join(words)}"

class XLNetContextEncoder(nn.Module):
    def __init__(self, xlnet_model, hidden_size, projection_dim):
        super().__init__()
        self.xlnet      = xlnet_model
        self.projection = nn.Sequential(
            nn.Linear(hidden_size, projection_dim),
            nn.LayerNorm(projection_dim),
            nn.GELU(),
        )
    def forward(self, input_ids, attention_mask, token_type_ids):
        outputs       = self.xlnet(input_ids=input_ids, attention_mask=attention_mask, token_type_ids=token_type_ids)
        hidden        = outputs.last_hidden_state
        mask_expanded = attention_mask.unsqueeze(-1).float()
        sum_hidden    = (hidden * mask_expanded).sum(dim=1)
        token_counts  = mask_expanded.sum(dim=1).clamp(min=1e-9)
        pooled        = sum_hidden / token_counts
        return self.projection(pooled)

## 2. New Cross-Attention Feature Fusion

In [3]:
class CrossModalityAttention(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.query_proj = nn.Linear(dim, dim)
        self.key_proj = nn.Linear(dim, dim)
        self.value_proj = nn.Linear(dim, dim)
        self.scale = dim ** -0.5
    def forward(self, query_feat, key_value_feat):
        q = self.query_proj(query_feat).unsqueeze(1)
        k = self.key_proj(key_value_feat).unsqueeze(1)
        v = self.value_proj(key_value_feat).unsqueeze(1)
        attn_scores = torch.bmm(q, k.transpose(1, 2)) * self.scale
        attn_probs = torch.softmax(attn_scores, dim=-1)
        compound_state = torch.bmm(attn_probs, v).squeeze(1)
        return compound_state

class InformationAggregator(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.ideology_ctx = CrossModalityAttention(dim)
        self.topic_ctx = CrossModalityAttention(dim)
        self.issue_ctx = CrossModalityAttention(dim)
    def forward(self, news_feat, id_feat, top_feat, iss_feat):
        id_compound = self.ideology_ctx(news_feat, id_feat)
        top_compound = self.topic_ctx(news_feat, top_feat)
        iss_compound = self.issue_ctx(news_feat, iss_feat)
        fused_embedding = news_feat + id_compound + top_compound + iss_compound
        return fused_embedding

## 3. Integrated Model & Dataset

In [4]:
STANCE_TO_IDEOLOGY = {
    "left" : "far left leaning ideology",
    'lean left' : 'slightly left leaning ideology',
    'center'    : 'centrist ideology with balanced reporting',
    'lean right': 'slightly right leaning ideology',
    'right'     : 'far right leaning ideology',
    'mixed'     : 'mixed or undefined ideological leaning',
    'not rated' : 'ideology not rated or unknown',
}
LABEL2ID = {"liberal": 0, "center": 1, "conservative": 2}

class UnifiedBiasDataset(Dataset):
    def __init__(self, df_or_path, xlnet_tokenizer, clip_tokenizer, max_len_xlnet=64, max_len_clip=77,
                 split="train", val_frac=0.1, test_frac=0.1, seed=42):
        self.xlnet_tokenizer = xlnet_tokenizer
        self.clip_tokenizer = clip_tokenizer
        self.max_len_xlnet = max_len_xlnet
        self.max_len_clip = max_len_clip
        if isinstance(df_or_path, str):
            df = pd.read_csv(df_or_path)
        else:
            df = df_or_path
        df = df.dropna(subset=['Title', 'News_Body', 'Label'])
        rng = np.random.default_rng(seed)
        idx = rng.permutation(len(df))
        n_test = int(len(df) * test_frac)
        n_val = int(len(df) * val_frac)
        if split == "test":
            idx = idx[:n_test]
        elif split == "val":
            idx = idx[n_test: n_test + n_val]
        else:
            idx = idx[n_test + n_val:]
        self.df = df.iloc[idx].reset_index(drop=True)
        self.labels = [LABEL2ID.get(str(l).strip().lower(), 1) for l in self.df['Label'].tolist()]
        self.issues = self.df['issue'].fillna('unknown issue').tolist()
        self.topics = self.df['topic'].fillna('unknown topic').tolist()
        self.stances = self.df['Stance'].fillna('center').tolist()
        self.titles = self.df['Title'].fillna('').tolist()
        self.news_bodies = self.df['News_Body'].fillna('').tolist()

    def __len__(self):
        return len(self.labels)

    def _tokenize_xlnet(self, text):
        return self.xlnet_tokenizer(text, max_length=self.max_len_xlnet, padding='max_length', truncation=True, return_tensors='pt')

    def _tokenize_clip(self, text):
        prompt = build_clip_prompt(text)
        return self.clip_tokenizer(prompt, max_length=self.max_len_clip, padding='max_length', truncation=True, return_tensors='pt')

    def __getitem__(self, idx):
        ideology_text = STANCE_TO_IDEOLOGY.get(str(self.stances[idx]).strip().lower(), 'ideology not rated or unknown')
        issue_enc = self._tokenize_xlnet(self.issues[idx])
        topic_enc = self._tokenize_xlnet(self.topics[idx])
        ideology_enc = self._tokenize_xlnet(ideology_text)
        news_text = f"{self.titles[idx]}. {self.news_bodies[idx]}"
        news_enc = self._tokenize_clip(news_text)
        return {
            'issue_input_ids': issue_enc['input_ids'].squeeze(0),
            'issue_attn_mask': issue_enc['attention_mask'].squeeze(0),
            'issue_token_type': issue_enc.get('token_type_ids', torch.zeros(self.max_len_xlnet, dtype=torch.long)).squeeze(0),
            'topic_input_ids': topic_enc['input_ids'].squeeze(0),
            'topic_attn_mask': topic_enc['attention_mask'].squeeze(0),
            'topic_token_type': topic_enc.get('token_type_ids', torch.zeros(self.max_len_xlnet, dtype=torch.long)).squeeze(0),
            'ideology_input_ids': ideology_enc['input_ids'].squeeze(0),
            'ideology_attn_mask': ideology_enc['attention_mask'].squeeze(0),
            'ideology_token_type': ideology_enc.get('token_type_ids', torch.zeros(self.max_len_xlnet, dtype=torch.long)).squeeze(0),
            'news_input_ids': news_enc['input_ids'].squeeze(0),
            'news_attn_mask': news_enc['attention_mask'].squeeze(0),
            'label': torch.tensor(self.labels[idx], dtype=torch.long)
        }

class IntegratedBiasDetectionModel(nn.Module):
    def __init__(self, projection_dim=256, num_labels=3):
        super().__init__()
        self.xlnet_backbone = XLNetModel.from_pretrained("xlnet-base-cased")
        hidden_xlnet = self.xlnet_backbone.config.d_model
        self.issue_encoder = XLNetContextEncoder(self.xlnet_backbone, hidden_xlnet, projection_dim)
        self.topic_encoder = XLNetContextEncoder(self.xlnet_backbone, hidden_xlnet, projection_dim)
        self.ideology_encoder = XLNetContextEncoder(self.xlnet_backbone, hidden_xlnet, projection_dim)
        self.news_feature_extractor = NewsFeatureExtractor(proj_dim=projection_dim)
        self.fusion_aggr = InformationAggregator(projection_dim)
        self.classifier = nn.Sequential(
            nn.Linear(projection_dim, 128),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(128, num_labels)
        )
    def forward(self, batch):
        iss_feat = self.issue_encoder(batch['issue_input_ids'], batch['issue_attn_mask'], batch['issue_token_type'])
        top_feat = self.topic_encoder(batch['topic_input_ids'], batch['topic_attn_mask'], batch['topic_token_type'])
        id_feat = self.ideology_encoder(batch['ideology_input_ids'], batch['ideology_attn_mask'], batch['ideology_token_type'])
        news_feat = self.news_feature_extractor(batch['news_input_ids'], batch['news_attn_mask'])
        fused_embedding = self.fusion_aggr(news_feat, id_feat, top_feat, iss_feat)
        logits = self.classifier(fused_embedding)
        return logits, fused_embedding

## 4. Training & Evaluation Logic

In [5]:
def train_model(model, train_loader, val_loader, device, epochs=1, lr=3e-4, save_path="integrated_model.pt"):
    optimizer = torch.optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=lr)
    criterion = nn.CrossEntropyLoss()
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    history = {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": []}
    best_val_acc = 0.0
    for epoch in range(1, epochs + 1):
        model.train()
        t_loss, t_correct, t_total = 0.0, 0, 0
        for batch in train_loader:
            batch = {k: v.to(device) for k, v in batch.items() if isinstance(v, torch.Tensor)}
            optimizer.zero_grad()
            logits, _ = model(batch)
            loss = criterion(logits, batch['label'])
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            t_loss += loss.item() * len(batch['label'])
            t_correct += (logits.argmax(1) == batch['label']).sum().item()
            t_total += len(batch['label'])
        scheduler.step()
        model.eval()
        v_loss, v_correct, v_total = 0.0, 0, 0
        with torch.no_grad():
            for batch in val_loader:
                batch = {k: v.to(device) for k, v in batch.items() if isinstance(v, torch.Tensor)}
                logits, _ = model(batch)
                loss = criterion(logits, batch['label'])
                v_loss += loss.item() * len(batch['label'])
                v_correct += (logits.argmax(1) == batch['label']).sum().item()
                v_total += len(batch['label'])
        t_acc = t_correct / t_total
        v_acc = v_correct / v_total
        history["train_loss"].append(t_loss / t_total)
        history["val_loss"].append(v_loss / v_total)
        history["train_acc"].append(t_acc)
        history["val_acc"].append(v_acc)
        log.info(f"Epoch {epoch}/{epochs} | Train loss: {t_loss/t_total:.4f} acc: {t_acc:.3f} | Val loss: {v_loss/v_total:.4f} acc: {v_acc:.3f}")
        if v_acc > best_val_acc:
            best_val_acc = v_acc
            torch.save(model.state_dict(), save_path)
    return history

def viz_training_curves(history):
    epochs = range(1, len(history["train_loss"]) + 1)
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))
    ax1.plot(epochs, history["train_loss"], "o-", label="Train")
    ax1.plot(epochs, history["val_loss"], "s--", label="Val")
    ax1.set_title("Loss Curves"); ax1.legend(); ax1.grid(alpha=0.3)
    ax2.plot(epochs, history["train_acc"], "o-", label="Train")
    ax2.plot(epochs, history["val_acc"], "s--", label="Val")
    ax2.set_title("Accuracy Curves"); ax2.set_ylim(0, 1); ax2.legend(); ax2.grid(alpha=0.3)
    plt.show()

def evaluate_test_set(model, test_loader, device, label_names=["liberal", "center", "conservative"]):
    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for batch in test_loader:
            batch = {k: v.to(device) for k, v in batch.items() if isinstance(v, torch.Tensor)}
            logits, _ = model(batch)
            all_preds += logits.argmax(1).cpu().tolist()
            all_labels += batch['label'].cpu().tolist()
    print(classification_report(all_labels, all_preds, target_names=label_names))
    cm = confusion_matrix(all_labels, all_preds)
    plt.figure(figsize=(7, 5))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=label_names, yticklabels=label_names)
    plt.title("Confusion Matrix")
    plt.show()

## 5. Execution

In [ ]:
CSV_PATH = "cleaned_political_bias_data (1).csv"
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

if os.path.exists(CSV_PATH):
    xn_tok = XLNetTokenizer.from_pretrained("xlnet-base-cased")
    cl_tok = CLIPTokenizer.from_pretrained("openai/clip-vit-base-patch32")
    train_ds = UnifiedBiasDataset(CSV_PATH, xn_tok, cl_tok, split="train")
    val_ds   = UnifiedBiasDataset(CSV_PATH, xn_tok, cl_tok, split="val")
    test_ds  = UnifiedBiasDataset(CSV_PATH, xn_tok, cl_tok, split="test")
    train_loader = DataLoader(train_ds, batch_size=16, shuffle=True)
    val_loader   = DataLoader(val_ds,   batch_size=16, shuffle=False)
    test_loader  = DataLoader(test_ds,  batch_size=16, shuffle=False)
    model = IntegratedBiasDetectionModel(projection_dim=256).to(DEVICE)
    history = train_model(model, train_loader, val_loader, DEVICE, epochs=1)
    viz_training_curves(history)
    evaluate_test_set(model, test_loader, DEVICE)
else:
    print("Dataset not found! Please check path.")

02:16:28 | INFO | HTTP Request: HEAD https://huggingface.co/xlnet-base-cased/resolve/main/tokenizer_config.json "HTTP/1.1 404 Not Found"
02:16:29 | INFO | HTTP Request: GET https://huggingface.co/api/models/xlnet-base-cased/tree/main/additional_chat_templates?recursive=false&expand=false "HTTP/1.1 307 Temporary Redirect"
02:16:29 | INFO | HTTP Request: GET https://huggingface.co/api/models/xlnet/xlnet-base-cased/tree/main/additional_chat_templates?recursive=false&expand=false "HTTP/1.1 404 Not Found"
02:16:29 | INFO | HTTP Request: GET https://huggingface.co/api/models/xlnet-base-cased/tree/main?recursive=true&expand=false "HTTP/1.1 307 Temporary Redirect"
02:16:30 | INFO | HTTP Request: GET https://huggingface.co/api/models/xlnet/xlnet-base-cased/tree/main?recursive=true&expand=false "HTTP/1.1 200 OK"
02:16:30 | INFO | HTTP Request: HEAD https://huggingface.co/xlnet-base-cased/resolve/main/spiece.model "HTTP/1.1 200 OK"
02:16:31 | INFO | HTTP Request: HEAD https://huggingface.co/opena

Loading weights:   0%|          | 0/206 [00:00<?, ?it/s]

XLNetModel LOAD REPORT from: xlnet-base-cased
Key            | Status     |  | 
---------------+------------+--+-
lm_loss.bias   | UNEXPECTED |  | 
lm_loss.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
02:16:38 | INFO | HTTP Request: GET https://huggingface.co/api/models/xlnet/xlnet-base-cased/discussions?p=0 "HTTP/1.1 200 OK"
02:16:38 | INFO | HTTP Request: GET https://huggingface.co/api/models/xlnet-base-cased/commits/refs%2Fpr%2F1 "HTTP/1.1 307 Temporary Redirect"
02:16:38 | INFO | HTTP Request: HEAD https://huggingface.co/openai/clip-vit-base-patch32/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
02:16:38 | INFO | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/openai/clip-vit-base-patch32/3d74acf9a28c67741b2f4f2ea7635f0aaf6f0268/config.json "HTTP/1.1 200 OK"
02:16:38 | INFO | HTTP Request: GET https://huggingface.co/api/models/xlnet/xlnet-base-cased/c

Loading weights:   0%|          | 0/196 [00:00<?, ?it/s]

CLIPTextModel LOAD REPORT from: openai/clip-vit-base-patch32
Key                                                            | Status     |  | 
---------------------------------------------------------------+------------+--+-
vision_model.encoder.layers.{0...11}.self_attn.q_proj.bias     | UNEXPECTED |  | 
vision_model.embeddings.patch_embedding.weight                 | UNEXPECTED |  | 
vision_model.encoder.layers.{0...11}.self_attn.v_proj.bias     | UNEXPECTED |  | 
vision_model.encoder.layers.{0...11}.self_attn.out_proj.weight | UNEXPECTED |  | 
vision_model.embeddings.position_embedding.weight              | UNEXPECTED |  | 
vision_model.encoder.layers.{0...11}.self_attn.k_proj.weight   | UNEXPECTED |  | 
vision_model.encoder.layers.{0...11}.self_attn.q_proj.weight   | UNEXPECTED |  | 
vision_model.encoder.layers.{0...11}.mlp.fc2.weight            | UNEXPECTED |  | 
vision_model.encoder.layers.{0...11}.mlp.fc2.bias              | UNEXPECTED |  | 
vision_model.encoder.layers.{0...11}.